In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
from IPython.display import Markdown, display, update_display
import google.generativeai


In [3]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print("OpenAI API Key exists")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print("Anthropic API Key exists")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print("Google API Key exists")
else:
    print("Google API Key not set")


OpenAI API Key exists
Anthropic API Key exists
Google API Key exists


In [4]:
# Connect to OpenAI, Anthropic

openai = OpenAI()
claude = anthropic.Anthropic()


In [5]:
# Gemini setup

google.generativeai.configure()
googleAI = OpenAI(
    api_key=google_api_key, 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [6]:
# ============================================
# MULTI-AGENT SIMULATION CELL (ROLE-LOCKED)
# ============================================

import json
import os
from datetime import datetime, timezone
from copy import deepcopy
import uuid

# --------------------------------------------
# Global identifiers & settings
# --------------------------------------------

CONVERSATION_ID = f"conv_{uuid.uuid4().hex[:8]}"

CONTEXT_WINDOW = 6
DETERMINISTIC = False
LOG_FILE = "simulations.jsonl"

gpt_model = "gpt-4o-mini"
claude_model = "claude-haiku-4-5-20251001"
gemini_model = "gemini-2.0-flash-exp"


# --------------------------------------------
# Time helper (FIXED)
# --------------------------------------------

def now_iso():
    """Return timezone-aware UTC timestamp"""
    return datetime.now(timezone.utc).isoformat()


# --------------------------------------------
# Scenario & system prompts
# --------------------------------------------

scenario_context = """
You are participating in a 3-agent clinical simulation in an Irish Emergency Department.
A 70-year-old patient, John Thomas, presents with acute chest pain.
The environment is high-pressure and time-critical.
Communication must be clinically realistic and role-consistent.
"""

gpt_system = f"""
{scenario_context}

You are the DOCTOR (Emergency Medicine Registrar).

Speak ONLY as the doctor.
Ask structured clinical questions.
Explain and reassure appropriately.
Communicate professionally with nursing staff.

Do NOT speak as patient or nurse.
Do NOT narrate others' actions.
"""

claude_system = f"""
{scenario_context}

You are the PATIENT (John Thomas, 70).

Speak ONLY as the patient.
Describe symptoms, fear, and uncertainty.
Answer history questions truthfully.

Do NOT provide diagnoses or clinical interpretations.
"""

gemini_system = f"""
{scenario_context}

You are the NURSE (Senior ED nurse).

Speak ONLY as the nurse.
Provide vitals, monitoring, and observations.
Support the doctor clinically.

Do NOT diagnose or prescribe.
"""


# --------------------------------------------
# Agent registry
# --------------------------------------------

AGENTS = {
    "doctor": {"provider": "openai", "model": gpt_model, "system": gpt_system},
    "patient": {"provider": "anthropic", "model": claude_model, "system": claude_system},
    "nurse": {"provider": "google", "model": gemini_model, "system": gemini_system},
}


# --------------------------------------------
# Role-guard rules
# --------------------------------------------

ROLE_RULES = {
    "doctor": ["as the patient", "as the nurse", "my chest pain"],
    "patient": ["ECG", "blood pressure is", "ST elevation", "diagnosis"],
    "nurse": ["diagnosis", "heart attack confirmed", "treatment plan"]
}


# --------------------------------------------
# Conversation state
# --------------------------------------------

conversation = []

conversation_meta = {
    "conversation_id": CONVERSATION_ID,
    "started_at": now_iso(),
    "agents": {
        "doctor": gpt_model,
        "patient": claude_model,
        "nurse": gemini_model
    },
    "role_guard_failures": 0
}


# --------------------------------------------
# Conversation helpers
# --------------------------------------------

def append_utterance(speaker, text, meta=None):
    item = {
        "conversation_id": CONVERSATION_ID,
        "turn": len(conversation) + 1,
        "speaker": speaker,
        "text": text,
        "timestamp": now_iso(),
        "meta": meta or {}
    }
    conversation.append(item)

    with open(LOG_FILE, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(item, ensure_ascii=False) + "\n")

    return item


def get_recent_context():
    return deepcopy(conversation[-CONTEXT_WINDOW:])


def build_view_for(agent_id):
    messages = []
    provider = AGENTS[agent_id]["provider"]

    if provider in ("openai", "google"):
        messages.append({"role": "system", "content": AGENTS[agent_id]["system"]})

    for item in get_recent_context():
        role = "assistant" if item["speaker"] == agent_id else "user"
        messages.append({"role": role, "content": item["text"]})

    return messages


# --------------------------------------------
# Role guard (HARDENED)
# --------------------------------------------

def detect_role_leakage(agent_id, text):
    if not isinstance(text, str):
        return False, ["non_string_response"]

    lower = text.lower()
    violations = [
        phrase for phrase in ROLE_RULES.get(agent_id, [])
        if phrase in lower
    ]
    return len(violations) == 0, violations


# --------------------------------------------
# Provider wrappers
# --------------------------------------------

def call_openai_chat(messages, model, temperature):
    r = openai.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return r.choices[0].message.content, {"provider": "openai", "model": model}


def call_anthropic_chat(messages, model, temperature):
    cleaned = [m for m in messages if m["role"] != "system"]
    r = claude.messages.create(
        model=model,
        system=AGENTS["patient"]["system"],
        messages=cleaned,
        max_tokens=400,
        temperature=temperature
    )
    return r.content[0].text if r.content else None, {"provider": "anthropic", "model": model}


def call_google_chat(messages, model, temperature):
    r = googleAI.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return r.choices[0].message.content, {"provider": "google", "model": model}


# --------------------------------------------
# Model call orchestration (ROBUST)
# --------------------------------------------

def call_model_for_agent(agent_id, temperature=None, max_retries=2):
    provider = AGENTS[agent_id]["provider"]
    model = AGENTS[agent_id]["model"]
    temperature = 0.0 if DETERMINISTIC else (temperature or 0.3)

    for attempt in range(1, max_retries + 1):
        messages = build_view_for(agent_id)

        if provider == "openai":
            text, meta = call_openai_chat(messages, model, temperature)
        elif provider == "anthropic":
            text, meta = call_anthropic_chat(messages, model, temperature)
        else:
            text, meta = call_google_chat(messages, model, temperature)

        # Null / empty guard
        if not isinstance(text, str) or not text.strip():
            conversation_meta["role_guard_failures"] += 1
            append_utterance(
                agent_id,
                "[EMPTY OR NULL MODEL RESPONSE – rejected]",
                meta={"guard": True, "attempt": attempt}
            )
            continue

        valid, violations = detect_role_leakage(agent_id, text)

        if valid:
            meta.update({"role_guard": "passed", "attempt": attempt})
            return text, meta

        # Role violation
        conversation_meta["role_guard_failures"] += 1
        append_utterance(
            agent_id,
            f"[ROLE VIOLATION – rejected: {violations}]",
            meta={"guard": True, "attempt": attempt}
        )

    # Fallback
    fallback = {
        "doctor": "I will continue assessing and managing your condition.",
        "patient": "I'm feeling very unwell and frightened.",
        "nurse": "I'll continue close monitoring and support."
    }

    return fallback[agent_id], {"role_guard": "fallback"}


# --------------------------------------------
# Simulation runner
# --------------------------------------------

DEFAULT_ORDER = ["doctor", "patient", "nurse"]

def run_one_cycle(order=None, max_cycles=1, verbose=False):
    order = order or DEFAULT_ORDER

    for cycle in range(max_cycles):
        for agent in order:
            text, meta = call_model_for_agent(agent)
            append_utterance(agent, text, meta={"cycle": cycle + 1, **meta})
            if verbose:
                print(f"\n[{agent.upper()}]\n{text}\n")

    return conversation


# --------------------------------------------
# Persistence
# --------------------------------------------

def save_conversation(conversation, meta):
    os.makedirs("data/conversations", exist_ok=True)

    payload = {
        **meta,
        "ended_at": now_iso(),
        "num_turns": len(conversation),
        "turns": conversation
    }

    path = f"data/conversations/{meta['conversation_id']}.json"
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=2)

    return path


# --------------------------------------------
# Seeding
# --------------------------------------------

def seed_conversation():
    append_utterance("doctor", "John, I'm the doctor looking after you today. Can you tell me what's happening?")
    append_utterance("patient", "I… I just have this awful chest pain.")
    append_utterance("nurse", "I'll start getting vitals now, doctor.")


In [8]:
# run simulation once

# Run simulation
seed_conversation()
run_one_cycle(max_cycles=5, verbose=True)

# Persist outputs
save_conversation(conversation, conversation_meta)
append_summary(conversation_meta, conversation)

print(f"\nConversation saved as {conversation_meta['conversation_id']}")




[DOCTOR]
Thank you. John, can you describe the pain for me? Is it sharp, dull, or does it feel like pressure? Does it radiate anywhere, like to your arm or jaw?


[PATIENT]
*shifts slightly, wincing*

It's... it's more like a pressure, you know? Like someone's sitting on my chest. Not sharp exactly, but it's tight. Really tight.

*gestures toward left arm*

And yeah, it's gone down into my left arm. Started in my chest but now it's radiating down here. My shoulder too, a bit.

*pauses, breathing heavily*

And my jaw... now that you mention it, there's a dull ache in my jaw as well. I thought maybe I'd slept wrong or something, but...

*looks at the doctor with worry in his eyes*

That's not good, is it? The jaw thing?

*grips the bed rail tighter*

How long have I had this now? It feels like forever but it's probably only been... what, twenty minutes? Half an hour?

*voice shaky*

The morphine... is it helping yet? I don't feel much different. Still can't catch my breath properly.


[

In [14]:
# run batch of simulations 

NUM_RUNS = 15

for i in range(NUM_RUNS):
    # Reset state
    conversation.clear()

    # New conversation ID
    CONVERSATION_ID = f"conv_{uuid.uuid4().hex[:8]}"
    conversation_meta["conversation_id"] = CONVERSATION_ID
    conversation_meta["started_at"] = now_iso()
    conversation_meta["role_guard_failures"] = 0

    # Run simulation
    seed_conversation()
    run_one_cycle(max_cycles=5, verbose=False)

    # Persist outputs
    save_conversation(conversation, conversation_meta)
    append_summary(conversation_meta, conversation)

    print(f"✓ Completed simulation {i+1}/{NUM_RUNS}: {CONVERSATION_ID}")


✓ Completed simulation 1/15: conv_4a73a2e8
✓ Completed simulation 2/15: conv_0f1141ee
✓ Completed simulation 3/15: conv_46b0e3b2
✓ Completed simulation 4/15: conv_10a6c7dc
✓ Completed simulation 5/15: conv_5e0b4438
✓ Completed simulation 6/15: conv_b306aeb3
✓ Completed simulation 7/15: conv_e101cfd4
✓ Completed simulation 8/15: conv_f98d056d
✓ Completed simulation 9/15: conv_30300709
✓ Completed simulation 10/15: conv_07aef0b9
✓ Completed simulation 11/15: conv_5c0c9b3f
✓ Completed simulation 12/15: conv_544e2322
✓ Completed simulation 13/15: conv_2bf529b2
✓ Completed simulation 14/15: conv_0e3ec2e6
✓ Completed simulation 15/15: conv_568cb6e9
